# GMSC Scorecard — Tree Depth-1 Surrogate

Give Me Some Credit 데이터셋에 대해 블랙박스 Teacher → Depth-1 Surrogate → ScorecardModel → Scorecard 파이프라인을 실행한다.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from decentra.surrogate import TreeSurrogate
from decentra._utils import logit, sigmoid

## 1. 데이터 로딩

In [3]:
df = pd.read_csv('../.data/give_me_some_credit/cs-training.csv', index_col=0)
y = df['SeriousDlqin2yrs']
X = df.drop('SeriousDlqin2yrs', axis=1).select_dtypes(include=[np.number])
X = X.fillna(X.median())

print(f"samples: {X.shape[0]:,}, features: {X.shape[1]}, default rate: {y.mean():.2%}")
X.head()

samples: 150,000, features: 10, default rate: 6.68%


,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


## 2. Train / Val / Test 분할

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

print(f"Train: {len(X_tr):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

Train: 96,000  Val: 24,000  Test: 30,000


## 3. BaseModel (Black-box) 학습

In [5]:
model = lgb.LGBMClassifier(
    max_depth=-1, n_estimators=1000, learning_rate=0.05,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=50, verbose=-1, random_state=42, n_jobs=-1,
)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Teacher AUC (test): {auc:.4f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[92]	valid_0's binary_logloss: 0.17998
Teacher AUC (test): 0.8664


## 4. Surrogate (Depth-1) 학습

In [10]:
from decentra.surrogate import TreeSurrogate


In [12]:

y_logit_train = logit(model.predict_proba(X_train)[:, 1]) * 40 / np.log(2)
y_logit_val = logit(model.predict_proba(X_val)[:, 1]) * 40 / np.log(2)

surr = TreeSurrogate(max_depth=1)
surr.fit(X_train, y_logit_train, eval_set=(X_val, y_logit_val))

surr_auc = roc_auc_score(y_test, sigmoid(surr.predict(X_test)))
print(f"Surrogate AUC (test): {surr_auc:.4f}")

surr = TreeSurrogate(max_depth=1, monotone_detect_mode='auto')
surr.fit(X_train, y_logit_train, eval_set=(X_val, y_logit_val))

surr_auc = roc_auc_score(y_test, sigmoid(surr.predict(X_test)))
print(f"Surrogate AUC (test): {surr_auc:.4f}")

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 403.763
Surrogate AUC (test): 0.8647
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 473.803
Surrogate AUC (test): 0.8634


In [13]:
surr.model_.booster_.trees_to_dataframe().query("split_feature == 'RevolvingUtilizationOfUnsecuredLines'")['split_gain'].nunique()

127

## 5. ScorecardModel 생성

In [14]:
sm = surr.to_scorecard_model(X_train, feature_names=list(X.columns))
print(sm)
# ScorecardModel로 직접 scoring (surrogate 없이)
scores = sm.score(X_test)
preds = surr.predict(X_test)
print(f"ScorecardModel vs Surrogate max diff: {np.max(np.abs(scores - preds)):.10f}")

ScorecardModel(base=-202.1262, 10 features, 132 bins)
ScorecardModel vs Surrogate max diff: 0.0000000000


In [15]:
sm = surr.to_scorecard_model(X_train, feature_names=list(X.columns), 
                             min_bin_ratio=0.01, max_bins_per_feature=10, max_bins_criterion='mse', min_ratio_criterion='mse',
                             
                             )
print(sm)
# ScorecardModel로 직접 scoring (surrogate 없이)
scores = sm.score(X_test)
preds = surr.predict(X_test)
print(f"ScorecardModel vs Surrogate max diff: {np.max(np.abs(scores - preds)):.10f}")

ScorecardModel(base=-202.1262, 10 features, 61 bins)
ScorecardModel vs Surrogate max diff: 35.3810257697


## 6. Scorecard 평점표

In [16]:
sc = sm.scorecard(X_train, y_train)
sc.to_dataframe().head(15)

,평가 항목,구간,가중치 점수,건수,Target 건수,구성비,Target Rate,사유코드,사유코드 설명
0,RevolvingUtilizationOfUnsecuredLines,< 0.06467,-37.7919,43422,762,36.19%,1.75%,N001,부정 요인 1순위
1,,0.06467 ~ 0.1126,-31.5481,10582,231,8.82%,2.18%,N002,부정 요인 2순위
2,,0.1126 ~ 0.2202,-18.0803,13431,397,11.19%,2.96%,N005,부정 요인 5순위
3,,0.2202 ~ 0.2948,-3.1948,6512,245,5.43%,3.76%,N020,부정 요인 20순위
4,,0.2948 ~ 0.3994,12.1570,7326,376,6.11%,5.13%,P019,긍정 요인 19순위
5,,0.3994 ~ 0.4915,22.8739,5291,341,4.41%,6.44%,P011,긍정 요인 11순위
6,,0.4915 ~ 0.7047,42.2944,9768,933,8.14%,9.55%,P009,긍정 요인 9순위
7,,0.7047 ~ 0.8735,60.3675,6512,968,5.43%,14.86%,P006,긍정 요인 6순위
8,,0.8735 ~ 1,67.3593,14504,2775,12.09%,19.13%,P004,긍정 요인 4순위
9,,>= 1,92.4471,2652,993,2.21%,37.44%,P001,긍정 요인 1순위


## 7. 직렬화 (저장 / 불러오기)

In [ ]:
import json
from decentra.scorecard_model import ScorecardModel

# 저장
with open('../.data/gmsc_scorecard_model.json', 'w') as f:
    json.dump(sm.to_dict(), f, indent=2)

# 불러오기
with open('../.data/gmsc_scorecard_model.json') as f:
    sm_loaded = ScorecardModel.from_dict(json.load(f))

diff = np.max(np.abs(sm_loaded.score(X_test) - sm.score(X_test)))
print(f"Round-trip max diff: {diff:.10f}")
sm_loaded